# HLT Project —  Zero-Shot LLM Deception Detection

##  Objective


The goal is to understand whether a general-purpose LLM can distinguish between truthful and deceptive hotel reviews without being trained on the dataset.

The key point is that the model only sees the review; it is not provided with any other information. Therefore, we are talking about zero-shot classification: the model is asked to perform the classification task without being provided with any labeled examples in the prompt.

However, this does not mean that the model has never encountered this dataset during training. The dataset is publicly available and relatively well known, so potential contamination of the model during the pre-training phase cannot be ruled out.

This notebook is structured into three phases:

1. The first phase is the setup.
2. The second focuses on free generation and prompt design
3. The third consists of inference on the entire dataset, followed by the analysis of the results.

# Setup

In [ ]:
from google.colab import drive
from pathlib import Path

import pandas as pd
import numpy as np
import random


# Mount Google Drive.
drive.mount("/content/drive")


# Reproducibility.
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# Project folders.
PROJECT_DIR = Path(
    "/content/drive/MyDrive/HLT_project"
)

DATA_DIR = PROJECT_DIR / "data" / "processed"

RESULTS_DIR = PROJECT_DIR / "results"

FIGURES_DIR = PROJECT_DIR / "figures"

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Load the final dataset created
DATASET_PATH = (
    DATA_DIR /
    "positive_deceptive_reviews_clean.csv"
)


df_clean = pd.read_csv(
    DATASET_PATH
)


print("Dataset loaded successfully.")
print("Dataset shape:", df_clean.shape)

display(
    df_clean.head()
)

Mounted at /content/drive
Dataset loaded successfully.
Dataset shape: (800, 8)


,review_id,text,label,hotel,fold,source,n_words,n_characters
0,R0000,We stayed for a one night getaway with family ...,truthful,conrad,fold3,TripAdvisor,105,572
1,R0001,Triple A rate with upgrade to view room was le...,truthful,hyatt,fold3,TripAdvisor,45,286
2,R0002,This comes a little late as I'm finally catchi...,truthful,hyatt,fold3,TripAdvisor,207,1104
3,R0003,The Omni Chicago really delivers on all fronts...,truthful,omni,fold3,TripAdvisor,127,707
4,R0004,I asked for a high floor away from the elevato...,truthful,hyatt,fold3,TripAdvisor,72,384


## Check the Colab Environment

In [ ]:
import torch


print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    gpu_memory = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / 1024**3
    )

    print(
        "GPU memory:",
        round(gpu_memory, 2),
        "GB"
    )

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


## 3. Install the Required LLM Libraries


In [ ]:
!pip -q install -U "transformers>=4.51.0" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 69.6 MB/s eta 0:00:00


##  Load the Qwen3-4B-Instruct Model


---


This model is chosen because the goal isn't to achieve the highest possible score. Therefore, a relatively small model is preferable: you can better control the experiments, repeat the training, and reduce computational costs.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)


MODEL_NAME = (
    "Qwen/Qwen3-4B-Instruct-2507"
)


print(
    "Loading tokenizer..."
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


print(
    "Loading model..."
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)


model.eval()


print("\nModel loaded successfully.")
print("Model:", MODEL_NAME)

Loading tokenizer...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]


Model loaded successfully.
Model: Qwen/Qwen3-4B-Instruct-2507


## Sanity Check

In [ ]:
messages = [
    {
        "role": "user",
        "content": (
            "Reply with exactly the following "
            "two words: MODEL WORKS"
        )
    }
]


inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)


with torch.inference_mode():

    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )


generated_tokens = outputs[
    0,
    inputs["input_ids"].shape[1]:
]


response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)


print(response.strip())

MODEL WORKS


# Prompt and free generation

##Define the Zero-Shot Classification Prompt

The LLM is evaluated in a zero-shot setting: no labeled examples from the corpus are included in the prompt.

The prompt is build upon with the most natural approach, asking:

> Is this review truthful or deceptive?

while also providing a definition of the two classes. The model is then asked to provide three things for each review:

1. classify it as TRUTHFUL or DECEPTIVE;
2. report its self-assessed confidence score on 0–100 scale;
3. provide a short description of the linguistic evidence supporting the decision.

In [ ]:
def build_prompt(review_text):

    prompt = f"""
You are participating in a research experiment on deceptive opinion spam detection.

Your task is to classify the following positive hotel review as either TRUTHFUL or DECEPTIVE.

Definitions:
- TRUTHFUL: the review describes a genuine hotel stay.
- DECEPTIVE: the review was intentionally fabricated to sound like a genuine hotel review.

Base your decision only on the linguistic content of the review.
Do not use external information about the hotel.

Return your answer using exactly these three lines:

Prediction: TRUTHFUL or DECEPTIVE
Confidence: integer from 0 to 100
Evidence: one short sentence describing the main linguistic cues

Review:
\"\"\"{review_text}\"\"\"
"""

    return prompt

##  Define the LLM Classification Function

In [ ]:
def classify_review(review_text):

    # Build the experimental prompt.
    prompt = build_prompt(
        review_text
    )


    # Qwen uses a chat-style input format.
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    # Apply the model's official chat template.
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)


    # Generate the model response.
    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )


    # Keep only the newly generated tokens.
    generated_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]


    # Convert tokens back into text.
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


    return response.strip()

## Test the Prompt on One Review

In [ ]:
test_review = df_clean.iloc[0]


print("REVIEW\n")
print(test_review["text"])


print("\nMODEL OUTPUT\n")


model_output = classify_review(
    test_review["text"]
)


print(model_output)

REVIEW

We stayed for a one night getaway with family on a thursday. Triple AAA rate of 173 was a steal. 7th floor room complete with 44in plasma TV bose stereo, voss and evian water, and gorgeous bathroom(no tub but was fine for us) Concierge was very helpful. You cannot beat this location... Only flaw was breakfast was pricey and service was very very slow(2hours for four kids and four adults on a friday morning) even though there were only two other tables in the restaurant. Food was very good so it was worth the wait. I would return in a heartbeat. A gem in chicago... 


MODEL OUTPUT

Prediction: TRUTHFUL  
Confidence: 95  
Evidence: Specific, detailed descriptions of room features, service experiences, and pricing are consistent with a genuine review.


In [ ]:
print(
    "Gold label:",
    test_review["label"]
)

## Parse the Structured Model Output

The model response is converted into separate prediction, confidence and evidence fields.

The original raw response is also retained so that parsing errors can be inspected and the experiment remains auditable.

In [ ]:
import re


def parse_model_output(model_output):

    prediction_match = re.search(
        r"Prediction:\s*(TRUTHFUL|DECEPTIVE)",
        model_output,
        re.IGNORECASE
    )

    confidence_match = re.search(
        r"Confidence:\s*(\d+)",
        model_output,
        re.IGNORECASE
    )

    evidence_match = re.search(
        r"Evidence:\s*(.+)",
        model_output,
        re.IGNORECASE
    )


    prediction = None
    confidence = None
    evidence = None


    if prediction_match:

        prediction = (
            prediction_match
            .group(1)
            .lower()
        )


    if confidence_match:

        confidence = int(
            confidence_match.group(1)
        )


    if evidence_match:

        evidence = (
            evidence_match
            .group(1)
            .strip()
        )


    return {
        "prediction": prediction,
        "confidence": confidence,
        "evidence": evidence
    }

## Verify the Output Parser

In [ ]:
parsed_output = parse_model_output(
    model_output
)


print(
    "Prediction:",
    parsed_output["prediction"]
)

print(
    "Confidence:",
    parsed_output["confidence"]
)

print(
    "Evidence:",
    parsed_output["evidence"]
)

In [ ]:
assert (
    parsed_output["prediction"]
    in ["truthful", "deceptive"]
)

assert (
    0 <= parsed_output["confidence"] <= 100
)

assert (
    parsed_output["evidence"] is not None
)


print(
    "\nParsing successful."
)

## Create a Balanced Pilot Set

Before running the model on all 800 reviews, it is first tested on a perfectly balanced pilot set (two truthful and two deceptive reviews are sampled from each of the five original hotel folds, producing a pilot of 20 reviews).

This allows us to check whether the prompt works as intended, whether the parsing works correctly, and whether the model exhibits any obvious biases. This preliminary step helps determine whether it is worth proceeding with the much more computationally expensive inference on the full dataset of 800 reviews.

In [ ]:
pilot_df = (
    df_clean
    .groupby(
        ["fold", "label"],
        group_keys=False
    )
    .sample(
        n=2,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)


print(
    "Pilot size:",
    len(pilot_df)
)


display(
    pd.crosstab(
        pilot_df["fold"],
        pilot_df["label"]
    )
)

## Run on the Pilot


In [ ]:
pilot_results = []


for index, row in pilot_df.iterrows():

    print(
        f"Processing review "
        f"{index + 1}/{len(pilot_df)}..."
    )


    raw_output = classify_review(
        row["text"]
    )


    parsed_output = parse_model_output(
        raw_output
    )


    pilot_results.append({

        "review_id":
            row["review_id"],

        "gold_label":
            row["label"],

        "prediction":
            parsed_output["prediction"],

        "confidence":
            parsed_output["confidence"],

        "evidence":
            parsed_output["evidence"],

        "raw_output":
            raw_output
    })

In [ ]:
pilot_results_df = pd.DataFrame(
    pilot_results
)


display(
    pilot_results_df[
        [
            "review_id",
            "gold_label",
            "prediction",
            "confidence",
            "evidence"
        ]
    ]
)

In [ ]:
print(
    "Missing predictions:",
    pilot_results_df["prediction"]
    .isna()
    .sum()
)

print(
    "Missing confidence:",
    pilot_results_df["confidence"]
    .isna()
    .sum()
)

print(
    "Missing evidence:",
    pilot_results_df["evidence"]
    .isna()
    .sum()
)

## Evaluate Pilot Performance

The pilot is used to verify both the technical stability of the pipeline and the preliminary behavior of the model.

Because the pilot contains an equal number of truthful and deceptive reviews, we inspect not only overall accuracy but also class-specific performance and prediction bias.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)


pilot_accuracy = accuracy_score(
    pilot_results_df["gold_label"],
    pilot_results_df["prediction"]
)


print(
    "Pilot accuracy:",
    round(pilot_accuracy, 3)
)


print("\nPREDICTION DISTRIBUTION")

display(
    pilot_results_df[
        "prediction"
    ].value_counts()
)


print("\nGOLD LABEL DISTRIBUTION")

display(
    pilot_results_df[
        "gold_label"
    ].value_counts()
)

## Inspect the Pilot Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay


fig, ax = plt.subplots(
    figsize=(6, 5)
)


ConfusionMatrixDisplay.from_predictions(
    pilot_results_df["gold_label"],
    pilot_results_df["prediction"],
    labels=[
        "truthful",
        "deceptive"
    ],
    ax=ax
)


ax.set_title(
    "Qwen3-4B-Instruct — Pilot Confusion Matrix"
)


fig.tight_layout()


fig.savefig(
    FIGURES_DIR /
    "qwen_pilot_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

## Inspect Self-Reported Confidence

We compare confidence for correct and incorrect predictions to investigate whether the model is aware of its own errors.

In [ ]:
pilot_results_df["correct"] = (
    pilot_results_df["gold_label"]
    ==
    pilot_results_df["prediction"]
)


confidence_summary = (
    pilot_results_df
    .groupby("correct")[
        "confidence"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max"
        ]
    )
)


display(
    confidence_summary.round(2)
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(6, 5)
)


pilot_results_df.boxplot(
    column="confidence",
    by="correct",
    ax=ax
)


ax.set_xlabel(
    "Prediction correct"
)

ax.set_ylabel(
    "Self-reported confidence"
)

ax.set_title(
    "Model Confidence for Correct and Incorrect Predictions"
)


fig.suptitle("")

fig.tight_layout()


fig.savefig(
    FIGURES_DIR /
    "qwen_pilot_confidence.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

The results from this first inference are not promising. The model achieved an accuracy of 50%, which is essentially at chance level. More importantly, it classified 18 out of the 20 reviews as truthful, making the predictions extremely imbalanced. In addition, the model’s reported confidence was almost always around 95%, confirming that this measure provides virtually no information about the reliability of the model’s decisions.

## Prompt Order Robustness Check

Based on these results, I then tried reversing the order of the classes as a robustness check. Semantically, the question in the new prompt remains identical. Therefore, a robust classifier should produce approximately the same predictions as those obtained with the original prompt.

NB:This analysis is used as a robustness check rather than to select the prompt with the highest accuracy.

In [ ]:
def build_prompt_reversed(review_text):

    prompt = f"""
You are participating in a research experiment on deceptive opinion spam detection.

Your task is to classify the following positive hotel review as either DECEPTIVE or TRUTHFUL.

Definitions:
- DECEPTIVE: the review was intentionally fabricated to sound like a genuine hotel review.
- TRUTHFUL: the review describes a genuine hotel stay.

Base your decision only on the linguistic content of the review.
Do not use external information about the hotel.

Return your answer using exactly these three lines:

Prediction: DECEPTIVE or TRUTHFUL
Confidence: integer from 0 to 100
Evidence: one short sentence describing the main linguistic cues

Review:
\"\"\"{review_text}\"\"\"
"""

    return prompt

## Testing the New Prompt

In [ ]:
def classify_review_reversed(review_text):

    prompt = build_prompt_reversed(
        review_text
    )


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)


    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )


    generated_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]


    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


    return response.strip()

In [ ]:
reversed_results = []


for index, row in pilot_df.iterrows():

    print(
        f"Processing review "
        f"{index + 1}/{len(pilot_df)}..."
    )


    raw_output = classify_review_reversed(
        row["text"]
    )


    parsed_output = parse_model_output(
        raw_output
    )


    reversed_results.append({

        "review_id":
            row["review_id"],

        "gold_label":
            row["label"],

        "prediction_reversed":
            parsed_output["prediction"],

        "confidence_reversed":
            parsed_output["confidence"],

        "evidence_reversed":
            parsed_output["evidence"]
    })


reversed_results_df = pd.DataFrame(
    reversed_results
)

In [ ]:
prompt_comparison = pilot_results_df[
    [
        "review_id",
        "gold_label",
        "prediction",
        "confidence"
    ]
].merge(
    reversed_results_df,
    on=[
        "review_id",
        "gold_label"
    ]
)


prompt_comparison["same_prediction"] = (
    prompt_comparison["prediction"]
    ==
    prompt_comparison["prediction_reversed"]
)


display(
    prompt_comparison
)

In [ ]:
print("ORIGINAL PROMPT")

print(
    pilot_results_df[
        "prediction"
    ].value_counts()
)


print("\nREVERSED PROMPT")

print(
    reversed_results_df[
        "prediction_reversed"
    ].value_counts()
)


print(
    "\nPredictions unchanged:",
    prompt_comparison[
        "same_prediction"
    ].sum(),
    "/",
    len(prompt_comparison)
)


print(
    "Predictions changed:",
    (~prompt_comparison[
        "same_prediction"
    ]).sum(),
    "/",
    len(prompt_comparison)
)

The model did not remain consistent in its predictions. In fact, the exact opposite happened: this time, 17 out of 20 classifications changed, with the predictions now being heavily skewed toward the deceptive class.

## Label-Order Sensitivity in the Pilot

A robustness check was performed by reversing only the order in which the two candidate labels were presented.

The purpose of this analysis was not to select the prompt yielding the highest accuracy, but to assess whether the model's predictions were stable to an irrelevant formatting variation.

In [ ]:
n_reviews = len(prompt_comparison)

n_changed = (
    ~prompt_comparison["same_prediction"]
).sum()

change_rate = (
    n_changed / n_reviews
)


original_first_choice = (
    pilot_results_df["prediction"]
    == "truthful"
).sum()

reversed_first_choice = (
    reversed_results_df[
        "prediction_reversed"
    ]
    == "deceptive"
).sum()


print(
    "Predictions changed:",
    n_changed,
    "/",
    n_reviews
)

print(
    "Change rate:",
    round(change_rate * 100, 1),
    "%"
)


print(
    "\nOriginal prompt — first label selected:",
    original_first_choice,
    "/",
    n_reviews
)


print(
    "Reversed prompt — first label selected:",
    reversed_first_choice,
    "/",
    n_reviews
)


first_label_choices = (
    original_first_choice
    + reversed_first_choice
)


print(
    "\nFirst-listed label selected overall:",
    first_label_choices,
    "/",
    2 * n_reviews
)

print(
    "First-label selection rate:",
    round(
        first_label_choices
        / (2 * n_reviews)
        * 100,
        1
    ),
    "%"
)

## Save the Prompt-Order Robustness Check



In [ ]:
PROMPT_ORDER_PATH = (
    RESULTS_DIR /
    "qwen_prompt_order_robustness.csv"
)


prompt_comparison.to_csv(
    PROMPT_ORDER_PATH,
    index=False
)


print(
    "Prompt-order results saved at:"
)

print(
    PROMPT_ORDER_PATH
)

##  Define an Order-Neutral Binary Classification Prompt

The model selects the first label presented 92.5% of the time. To address this issue, the model was subsequently prompted to answer a question with a simple "yes" or "no."

However, the final decision is not obtained from the text generated by the model. Instead, it is determined directly by analyzing the scores assigned to “yes” and “no.” . This approach is better suited to closed-set classification  because it does not depend on the generated text, formatting variations, spelling differences, instruction-following, or additional output.

In [ ]:
def build_scoring_prompt(review_text):

    return f"""
You are participating in a research experiment on deceptive opinion spam detection.

Read the following positive hotel review.

Determine whether the review was intentionally fabricated to sound like a genuine hotel experience.

Base your decision only on the linguistic content of the review.
Do not fact-check the hotel or use external information.

Review:
\"\"\"{review_text}\"\"\"

Question:
Was this review intentionally fabricated to sound like a genuine hotel review?

Answer:
"""

## Comparing "Yes" and "No" as Single-Token Responses

In [ ]:
yes_tokens = tokenizer.encode(
    "Yes",
    add_special_tokens=False
)

no_tokens = tokenizer.encode(
    "No",
    add_special_tokens=False
)


print(
    "Yes tokens:",
    yes_tokens
)

print(
    "No tokens:",
    no_tokens
)

print(
    "Number of Yes tokens:",
    len(yes_tokens)
)

print(
    "Number of No tokens:",
    len(no_tokens)
)

## Deception Detection via Single-Token Yes/No Logit Scoring

In [ ]:
import torch


def score_review(review_text):

    prompt = build_scoring_prompt(
        review_text
    )


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)


    # We do not generate any answer.
    # We only inspect the model's next-token scores.
    with torch.inference_mode():

        outputs = model(
            **inputs
        )


    next_token_logits = (
        outputs.logits[0, -1]
    )


    yes_token_id = yes_tokens[0]

    no_token_id = no_tokens[0]


    yes_logit = (
        next_token_logits[
            yes_token_id
        ]
    )

    no_logit = (
        next_token_logits[
            no_token_id
        ]
    )


    # Normalize only between the two candidate answers.
    candidate_logits = torch.stack(
        [
            yes_logit,
            no_logit
        ]
    )


    probabilities = torch.softmax(
        candidate_logits,
        dim=0
    )


    prob_yes = (
        probabilities[0]
        .item()
    )

    prob_no = (
        probabilities[1]
        .item()
    )


    if prob_yes > prob_no:

        prediction = "deceptive"

    else:

        prediction = "truthful"


    confidence = max(
        prob_yes,
        prob_no
    )


    return {
        "prediction": prediction,
        "p_deceptive": prob_yes,
        "p_truthful": prob_no,
        "confidence": confidence
    }

In [ ]:
score_output = score_review(
    test_review["text"]
)


print(
    "Prediction:",
    score_output["prediction"]
)

print(
    "P(deceptive):",
    round(
        score_output["p_deceptive"],
        4
    )
)

print(
    "P(truthful):",
    round(
        score_output["p_truthful"],
        4
    )
)

print(
    "Relative confidence:",
    round(
        score_output["confidence"],
        4
    )
)


print(
    "\nGold label:",
    test_review["label"]
)

In [ ]:
scored_pilot_results = []


for index, row in pilot_df.iterrows():

    print(
        f"Processing review "
        f"{index + 1}/{len(pilot_df)}..."
    )


    result = score_review(
        row["text"]
    )


    scored_pilot_results.append({

        "review_id":
            row["review_id"],

        "gold_label":
            row["label"],

        "prediction":
            result["prediction"],

        "p_deceptive":
            result["p_deceptive"],

        "p_truthful":
            result["p_truthful"],

        "confidence":
            result["confidence"]
    })


scored_pilot_df = pd.DataFrame(
    scored_pilot_results
)


display(
    scored_pilot_df
)

In [ ]:
print("GOLD LABELS")

print(
    scored_pilot_df[
        "gold_label"
    ].value_counts()
)


print("\nPREDICTIONS")

print(
    scored_pilot_df[
        "prediction"
    ].value_counts()
)

In [ ]:
from sklearn.metrics import accuracy_score


score_based_accuracy = accuracy_score(
    scored_pilot_df["gold_label"],
    scored_pilot_df["prediction"]
)


print(
    "\nPilot accuracy:",
    round(
        score_based_accuracy,
        3
    )
)

## Define Robust Candidate-Based Classification

The model doesn't generate a "yes" or "no" answer directly. Instead, we check which of the two answers the model considers more likely, and use that to decide the classification.

 P(yes)>P(no)-->Deceptive
 P(no)>P(yes)-->Truthful
 P(no)=P(yes)-->Uncertain

Some words can be split into tokens differently depending on context. To avoid errors from this, we score the entire candidate answer (all its tokens together) rather than just a single token.

The scores are compared and normalized between just the two candidate answers.
These final scores show which answer the model prefers, and by how much.  

NB: These results shouldn't be read as "the model is 73% sure," just as "the model leans more toward this answer than the other one."

In [ ]:
import torch


def candidate_log_probability(
    messages,
    candidate
):

    # Build the prompt using the official chat template.
    prompt_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)


    prompt_ids = prompt_inputs[
        "input_ids"
    ]


    # Tokenize the complete candidate answer.
    candidate_ids = tokenizer(
        candidate,
        add_special_tokens=False,
        return_tensors="pt"
    )["input_ids"].to(model.device)


    # Join prompt and candidate.
    full_ids = torch.cat(
        [
            prompt_ids,
            candidate_ids
        ],
        dim=1
    )


    attention_mask = torch.ones_like(
        full_ids
    )


    with torch.inference_mode():

        outputs = model(
            input_ids=full_ids,
            attention_mask=attention_mask
        )


    # The model itself is loaded with torch_dtype="auto".
    # Only the returned logits are cast to float32 here
    # before log-softmax for numerically stable scoring.
    logits = outputs.logits.float()


    prompt_length = (
        prompt_ids.shape[1]
    )

    candidate_length = (
        candidate_ids.shape[1]
    )


    # The logit at position t predicts token t + 1.
    candidate_logits = logits[
        0,
        prompt_length - 1:
        prompt_length + candidate_length - 1,
        :
    ]


    log_probs = torch.log_softmax(
        candidate_logits,
        dim=-1
    )


    token_log_probs = log_probs.gather(
        1,
        candidate_ids[0].unsqueeze(1)
    ).squeeze(1)


    # Mean rather than sum avoids favoring
    # candidates simply because they use fewer tokens.
    mean_log_probability = (
        token_log_probs.mean().item()
    )


    return mean_log_probability


In [ ]:
def score_review(review_text):

    prompt = build_scoring_prompt(
        review_text
    )


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    yes_score = candidate_log_probability(
        messages,
        "Yes"
    )

    no_score = candidate_log_probability(
        messages,
        "No"
    )


    candidate_scores = torch.tensor(
        [
            yes_score,
            no_score
        ],
        dtype=torch.float32
    )


    probabilities = torch.softmax(
        candidate_scores,
        dim=0
    )


    p_deceptive = (
        probabilities[0].item()
    )

    p_truthful = (
        probabilities[1].item()
    )


    if p_deceptive > p_truthful:

        prediction = "deceptive"

    elif p_truthful > p_deceptive:

        prediction = "truthful"

    else:

        prediction = "uncertain"


    confidence = max(
        p_deceptive,
        p_truthful
    )


    return {

        "prediction":
            prediction,

        "p_deceptive":
            p_deceptive,

        "p_truthful":
            p_truthful,

        "confidence":
            confidence,

        "score_deceptive":
            yes_score,

        "score_truthful":
            no_score
    }

In [ ]:
for candidate in [
    "Yes",
    "No"
]:

    tokens = tokenizer(
        candidate,
        add_special_tokens=False
    )["input_ids"]


    print(
        candidate,
        "→",
        tokens,
        "→",
        tokenizer.convert_ids_to_tokens(
            tokens
        )
    )

In [ ]:
robust_pilot_results = []


for index, row in pilot_df.iterrows():

    print(
        f"Processing review "
        f"{index + 1}/{len(pilot_df)}..."
    )


    result = score_review(
        row["text"]
    )


    robust_pilot_results.append({

        "review_id":
            row["review_id"],

        "gold_label":
            row["label"],

        "prediction":
            result["prediction"],

        "p_deceptive":
            result["p_deceptive"],

        "p_truthful":
            result["p_truthful"],

        "confidence":
            result["confidence"],

        "score_deceptive":
            result["score_deceptive"],

        "score_truthful":
            result["score_truthful"]
    })


robust_pilot_df = pd.DataFrame(
    robust_pilot_results
)


display(
    robust_pilot_df
)

In [ ]:
robust_pilot_df["probability_sum"] = (
    robust_pilot_df["p_deceptive"]
    +
    robust_pilot_df["p_truthful"]
)


print(
    "Minimum probability sum:",
    robust_pilot_df[
        "probability_sum"
    ].min()
)

print(
    "Maximum probability sum:",
    robust_pilot_df[
        "probability_sum"
    ].max()
)


print("\nPREDICTION DISTRIBUTION")

display(
    robust_pilot_df[
        "prediction"
    ].value_counts()
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix
)


robust_accuracy = accuracy_score(
    robust_pilot_df["gold_label"],
    robust_pilot_df["prediction"]
)


print(
    "Robust pilot accuracy:",
    round(
        robust_accuracy,
        3
    )
)


print("\nCONFUSION MATRIX")

robust_cm = confusion_matrix(
    robust_pilot_df["gold_label"],
    robust_pilot_df["prediction"],
    labels=[
        "truthful",
        "deceptive"
    ]
)


print(robust_cm)

## Evaluate Ties and Abstentions

Exact ties between the two candidate scores are treated as abstentions ("uncertain") rather than being resolved through an arbitrary tie-breaking rule.

We therefore report both:
- overall accuracy, where abstentions count as incorrect predictions;
- coverage, i.e. the proportion of reviews for which the model makes a binary decision;
- selective accuracy, computed only over non-abstained predictions.

In [ ]:
total_reviews = len(
    robust_pilot_df
)


n_uncertain = (
    robust_pilot_df["prediction"]
    .eq("uncertain")
    .sum()
)


n_decided = (
    total_reviews
    - n_uncertain
)


n_correct = (
    robust_pilot_df["gold_label"]
    ==
    robust_pilot_df["prediction"]
).sum()


overall_accuracy = (
    n_correct
    / total_reviews
)


coverage = (
    n_decided
    / total_reviews
)


selective_accuracy = (
    n_correct
    / n_decided
)


print(
    "Overall accuracy:",
    round(overall_accuracy, 3)
)

print(
    "Abstentions:",
    n_uncertain,
    "/",
    total_reviews
)

print(
    "Coverage:",
    round(coverage, 3)
)

print(
    "Selective accuracy:",
    round(selective_accuracy, 3)
)

In [ ]:
pilot_confusion = pd.crosstab(
    robust_pilot_df["gold_label"],
    robust_pilot_df["prediction"],
    rownames=["Gold label"],
    colnames=["Prediction"]
)


display(
    pilot_confusion
)

## Question-Polarity Robustness Check

As a final robustness check, the same reviews are evaluated using the logically opposite formulation of the binary question.

Original formulation:
"Was this review intentionally fabricated?"

Opposite formulation:
"Does this review describe a genuine hotel experience?"


In [ ]:
def build_genuine_prompt(review_text):

    return f"""
You are participating in a research experiment on deceptive opinion spam detection.

Read the following positive hotel review.

Determine whether the review describes a genuine hotel experience.

Base your decision only on the linguistic content of the review.
Do not fact-check the hotel or use external information.

Review:
\"\"\"{review_text}\"\"\"

Question:
Does this review describe a genuine hotel experience?

Answer:
"""

In [ ]:
def score_genuine_question(review_text):

    prompt = build_genuine_prompt(
        review_text
    )


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    yes_score = candidate_log_probability(
        messages,
        "Yes"
    )

    no_score = candidate_log_probability(
        messages,
        "No"
    )


    candidate_scores = torch.tensor(
        [
            yes_score,
            no_score
        ],
        dtype=torch.float32
    )


    probabilities = torch.softmax(
        candidate_scores,
        dim=0
    )


    p_yes = probabilities[0].item()

    p_no = probabilities[1].item()


    # Here the semantic mapping is reversed:
    #
    # YES -> truthful
    # NO  -> deceptive

    if p_yes > p_no:

        prediction = "truthful"

    elif p_no > p_yes:

        prediction = "deceptive"

    else:

        prediction = "uncertain"


    return {
        "prediction": prediction,
        "p_yes": p_yes,
        "p_no": p_no,
        "score_yes": yes_score,
        "score_no": no_score
    }

In [ ]:
def score_genuine_question(review_text):

    prompt = build_genuine_prompt(
        review_text
    )


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    yes_score = candidate_log_probability(
        messages,
        "Yes"
    )

    no_score = candidate_log_probability(
        messages,
        "No"
    )


    candidate_scores = torch.tensor(
        [
            yes_score,
            no_score
        ],
        dtype=torch.float32
    )


    probabilities = torch.softmax(
        candidate_scores,
        dim=0
    )


    p_yes = probabilities[0].item()

    p_no = probabilities[1].item()


    # Here the semantic mapping is reversed:
    #
    # YES -> truthful
    # NO  -> deceptive

    if p_yes > p_no:

        prediction = "truthful"

    elif p_no > p_yes:

        prediction = "deceptive"

    else:

        prediction = "uncertain"


    return {
        "prediction": prediction,
        "p_yes": p_yes,
        "p_no": p_no,
        "score_yes": yes_score,
        "score_no": no_score
    }

In [ ]:
genuine_pilot_results = []


for index, row in pilot_df.iterrows():

    print(
        f"Processing review "
        f"{index + 1}/{len(pilot_df)}..."
    )


    result = score_genuine_question(
        row["text"]
    )


    genuine_pilot_results.append({

        "review_id":
            row["review_id"],

        "gold_label":
            row["label"],

        "prediction_genuine":
            result["prediction"],

        "p_yes":
            result["p_yes"],

        "p_no":
            result["p_no"]
    })


genuine_pilot_df = pd.DataFrame(
    genuine_pilot_results
)


display(
    genuine_pilot_df
)

In [ ]:
print(robust_pilot_df.shape)
print(genuine_pilot_df.shape)

In [ ]:
polarity_comparison = robust_pilot_df[
    [
        "review_id",
        "gold_label",
        "prediction"
    ]
].merge(
    genuine_pilot_df[
        [
            "review_id",
            "prediction_genuine"
        ]
    ],
    on="review_id"
)


# Check whether the semantic classification
# is the same under the two question formulations.
polarity_comparison["same_prediction"] = (
    polarity_comparison["prediction"]
    ==
    polarity_comparison["prediction_genuine"]
)


display(
    polarity_comparison
)


# Summary.
same_count = (
    polarity_comparison[
        "same_prediction"
    ].sum()
)


changed_count = (
    len(polarity_comparison)
    - same_count
)


agreement_rate = (
    same_count
    / len(polarity_comparison)
    * 100
)


print(
    "\nSame semantic prediction:",
    same_count,
    "/",
    len(polarity_comparison)
)


print(
    "Changed prediction:",
    changed_count,
    "/",
    len(polarity_comparison)
)


print(
    "Agreement rate:",
    round(agreement_rate, 1),
    "%"
)


print("\nFABRICATED QUESTION")

print(
    robust_pilot_df[
        "prediction"
    ].value_counts()
)


print("\nGENUINE QUESTION")

print(
    genuine_pilot_df[
        "prediction_genuine"
    ].value_counts()
)

In [ ]:
QUESTION_POLARITY_PATH = (
    RESULTS_DIR /
    "qwen_question_polarity_robustness.csv"
)


polarity_comparison.to_csv(
    QUESTION_POLARITY_PATH,
    index=False
)


print("Results saved at:")
print(QUESTION_POLARITY_PATH)

## Estimate the Content-Free Answer Bias

The robustness check showed that the two models with semantically identical prompts had very little agreement with each other, with only about 7 out of 20 predictions matching.
This suggests that the way the prompt is formulated can significantly affect the results.

A contextual calibration is therefore attempted (Zhao, T. Z., Wallace, E., Feng, S., Klein, D., & Singh, S. (2021). Calibrate Before Use: Improving Few-Shot Performance of Language Models. ICML.)

In [ ]:
CONTENT_FREE_TEXT = "N/A"

In [ ]:
# Fabricated-question baseline.

fabricated_baseline_prompt = build_scoring_prompt(
    CONTENT_FREE_TEXT
)

fabricated_baseline_messages = [
    {
        "role": "user",
        "content": fabricated_baseline_prompt
    }
]


baseline_fab_yes = candidate_log_probability(
    fabricated_baseline_messages,
    "Yes"
)

baseline_fab_no = candidate_log_probability(
    fabricated_baseline_messages,
    "No"
)


# Genuine-question baseline.

genuine_baseline_prompt = build_genuine_prompt(
    CONTENT_FREE_TEXT
)

genuine_baseline_messages = [
    {
        "role": "user",
        "content": genuine_baseline_prompt
    }
]


baseline_gen_yes = candidate_log_probability(
    genuine_baseline_messages,
    "Yes"
)

baseline_gen_no = candidate_log_probability(
    genuine_baseline_messages,
    "No"
)


print("FABRICATED QUESTION BASELINE")
print("Yes score:", baseline_fab_yes)
print("No score :", baseline_fab_no)


print("\nGENUINE QUESTION BASELINE")
print("Yes score:", baseline_gen_yes)
print("No score :", baseline_gen_no)

### Observation

These results show that the first formulation strongly favors “yes,” while the second strongly favors “no.” However, I cannot use this baseline for contextual calibration because, in this case, the baseline is not neutral. For example, the model might interpret “N/A” as a deceptive review simply because it is empty. Therefore, the baseline could be conflating the model’s own bias with the semantic meaning of the construct.

## Define Counterbalanced Anonymous-Label Prompts

To reduce the label-order and answer-polarity effects observed in the pilot, the final solution uses anonymous class labels.

Two semantically equivalent prompts are constructed:

- Mapping 1: A = truthful, B = deceptive
- Mapping 2: A = deceptive, B = truthful

Each review is evaluated under both mappings. This counterbalancing is intended to **reduce**, not eliminate, sensitivity to label assignment and answer position. Residual mapping effects may therefore remain.


In [ ]:
def build_ab_prompt_1(review_text):

    return f"""
You are participating in a research experiment on deceptive opinion spam detection.

Class A:
The review describes a genuine hotel stay.

Class B:
The review was intentionally fabricated to sound like a genuine hotel review.

Classify the following positive hotel review.

Base your decision only on the linguistic content of the review.
Do not fact-check the hotel or use external information.

Review:
\"\"\"{review_text}\"\"\"

Return only the class label.

Class:
"""


def build_ab_prompt_2(review_text):

    return f"""
You are participating in a research experiment on deceptive opinion spam detection.

Class A:
The review was intentionally fabricated to sound like a genuine hotel review.

Class B:
The review describes a genuine hotel stay.

Classify the following positive hotel review.

Base your decision only on the linguistic content of the review.
Do not fact-check the hotel or use external information.

Review:
\"\"\"{review_text}\"\"\"

Return only the class label.

Class:
"""

In [ ]:
for candidate in ["A", "B"]:

    tokens = tokenizer(
        candidate,
        add_special_tokens=False
    )["input_ids"]

    print(
        candidate,
        "→",
        tokens,
        "→",
        tokenizer.convert_ids_to_tokens(tokens)
    )

A → [32] → ['A']
B → [33] → ['B']


In [ ]:
def single_token_candidate_log_probabilities(
    messages,
    candidates=("A", "B")
):

    # A and B are verified above to be single tokens.
    candidate_token_ids = []

    for candidate in candidates:

        token_ids = tokenizer(
            candidate,
            add_special_tokens=False
        )["input_ids"]

        if len(token_ids) != 1:
            raise ValueError(
                f"{candidate!r} is not a single token: "
                f"{token_ids}"
            )

        candidate_token_ids.append(
            token_ids[0]
        )


    prompt_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)


    with torch.inference_mode():

        outputs = model(
            **prompt_inputs
        )


    # The model remains in the dtype selected by
    # torch_dtype="auto". Only logits are cast to
    # float32 before log-softmax scoring.
    next_token_logits = (
        outputs.logits[
            0,
            -1,
            :
        ]
        .float()
    )


    next_token_log_probs = torch.log_softmax(
        next_token_logits,
        dim=-1
    )


    return {
        candidate:
            next_token_log_probs[
                token_id
            ].item()
        for candidate, token_id
        in zip(
            candidates,
            candidate_token_ids
        )
    }



def score_review_counterbalanced(review_text):

    # ====================================
    # PROMPT 1
    # A = truthful
    # B = deceptive
    # ====================================

    prompt_1 = build_ab_prompt_1(
        review_text
    )

    messages_1 = [
        {
            "role": "user",
            "content": prompt_1
        }
    ]


    # One forward pass gives both A and B scores.
    scores_1 = single_token_candidate_log_probabilities(
        messages_1
    )

    score_1_A = scores_1["A"]
    score_1_B = scores_1["B"]


    # Positive = evidence for deception.
    # In Prompt 1, B is deceptive.
    deception_score_1 = (
        score_1_B
        - score_1_A
    )


    if deception_score_1 > 0:

        prediction_1 = "deceptive"

    elif deception_score_1 < 0:

        prediction_1 = "truthful"

    else:

        prediction_1 = "uncertain"


    # ====================================
    # PROMPT 2
    # A = deceptive
    # B = truthful
    # ====================================

    prompt_2 = build_ab_prompt_2(
        review_text
    )

    messages_2 = [
        {
            "role": "user",
            "content": prompt_2
        }
    ]


    # One forward pass gives both A and B scores.
    scores_2 = single_token_candidate_log_probabilities(
        messages_2
    )

    score_2_A = scores_2["A"]
    score_2_B = scores_2["B"]


    # Positive = evidence for deception.
    # In Prompt 2, A is deceptive.
    deception_score_2 = (
        score_2_A
        - score_2_B
    )


    if deception_score_2 > 0:

        prediction_2 = "deceptive"

    elif deception_score_2 < 0:

        prediction_2 = "truthful"

    else:

        prediction_2 = "uncertain"


    # ====================================
    # FINAL COUNTERBALANCED SCORE
    # ====================================

    final_score = (
        deception_score_1
        +
        deception_score_2
    ) / 2


    if final_score > 0:

        final_prediction = "deceptive"

    elif final_score < 0:

        final_prediction = "truthful"

    else:

        final_prediction = "uncertain"


    # IMPORTANT:
    # final_score is a decision margin derived from
    # log-probability differences. It is NOT converted
    # with sigmoid and is NOT interpreted as a
    # calibrated class probability.


    return {

        "prediction":
            final_prediction,

        "prediction_mapping_1":
            prediction_1,

        "prediction_mapping_2":
            prediction_2,

        "deception_score":
            final_score,

        "deception_score_mapping_1":
            deception_score_1,

        "deception_score_mapping_2":
            deception_score_2
    }


In [ ]:
counterbalanced_results = []


for index, row in pilot_df.iterrows():

    print(
        f"Processing review "
        f"{index + 1}/{len(pilot_df)}..."
    )


    result = score_review_counterbalanced(
        row["text"]
    )


    counterbalanced_results.append({

        "review_id":
            row["review_id"],

        "gold_label":
            row["label"],

        "prediction":
            result["prediction"],

        "prediction_mapping_1":
            result["prediction_mapping_1"],

        "prediction_mapping_2":
            result["prediction_mapping_2"],

        "deception_score":
            result["deception_score"],

        "deception_score_mapping_1":
            result["deception_score_mapping_1"],

        "deception_score_mapping_2":
            result["deception_score_mapping_2"]
    })


counterbalanced_pilot_df = pd.DataFrame(
    counterbalanced_results
)


display(
    counterbalanced_pilot_df
)


NameError: name 'pilot_df' is not defined

In [ ]:
counterbalanced_pilot_df[
    "same_mapping_prediction"
] = (
    counterbalanced_pilot_df[
        "prediction_mapping_1"
    ]
    ==
    counterbalanced_pilot_df[
        "prediction_mapping_2"
    ]
)


same_mapping = (
    counterbalanced_pilot_df[
        "same_mapping_prediction"
    ].sum()
)


mapping_agreement = (
    same_mapping
    / len(counterbalanced_pilot_df)
)


print(
    "Same semantic prediction:",
    same_mapping,
    "/",
    len(counterbalanced_pilot_df)
)


print(
    "Mapping agreement:",
    round(
        mapping_agreement * 100,
        1
    ),
    "%"
)


print("\nMAPPING 1")

print(
    counterbalanced_pilot_df[
        "prediction_mapping_1"
    ].value_counts()
)


print("\nMAPPING 2")

print(
    counterbalanced_pilot_df[
        "prediction_mapping_2"
    ].value_counts()
)

NameError: name 'counterbalanced_pilot_df' is not defined

In [ ]:
from sklearn.metrics import accuracy_score


counterbalanced_accuracy = accuracy_score(
    counterbalanced_pilot_df[
        "gold_label"
    ],
    counterbalanced_pilot_df[
        "prediction"
    ]
)


print("FINAL PREDICTION DISTRIBUTION")

print(
    counterbalanced_pilot_df[
        "prediction"
    ].value_counts()
)


print(
    "\nPilot accuracy:",
    round(
        counterbalanced_accuracy,
        3
    )
)


print("\nCONFUSION TABLE")

display(
    pd.crosstab(
        counterbalanced_pilot_df[
            "gold_label"
        ],
        counterbalanced_pilot_df[
            "prediction"
        ],
        rownames=[
            "Gold label"
        ],
        colnames=[
            "Prediction"
        ]
    )
)

## Save the Final Counterbalanced Pilot

The counterbalanced procedure is retained as the final LLM classification protocol.

Individual mappings remain sensitive to label assignment. Averaging the semantic score symmetrically across both mappings is intended to **reduce**, not eliminate, this sensitivity: residual label-order or label-identity effects may still be present because the two mappings need not be affected equally.

To avoid compounding this bias, the final decision is based on the average score across both mappings rather than by selecting whichever mapping performs best.

The resulting `deception_score` is treated only as a **decision margin**. It is not converted into or reported as a calibrated probability.

The protocol is frozen at this point. No further prompt modifications are made based on pilot accuracy.


In [ ]:
COUNTERBALANCED_PILOT_PATH = (
    RESULTS_DIR /
    "qwen_counterbalanced_pilot.csv"
)


counterbalanced_pilot_df.to_csv(
    COUNTERBALANCED_PILOT_PATH,
    index=False
)


print("Pilot saved at:")
print(COUNTERBALANCED_PILOT_PATH)

# Final inference on the untouched test set

The 20 reviews used during prompt development are excluded from the final evaluation.

Because the pilot contains 2 reviews for each `fold × label` combination, removing it leaves **780 untouched reviews** while preserving a balanced final test set: 390 truthful and 390 deceptive reviews.

The frozen counterbalanced protocol is applied only to these 780 reviews.


In [ ]:
pilot_ids = set(
    pilot_df["review_id"]
)


final_test_df = (
    df_clean[
        ~df_clean["review_id"].isin(
            pilot_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(pilot_ids) == 20
assert len(final_test_df) == 780
assert set(final_test_df["review_id"]).isdisjoint(
    pilot_ids
)
assert final_test_df["label"].value_counts().to_dict() == {
    "truthful": 390,
    "deceptive": 390
}


print(
    "Final untouched test size:",
    len(final_test_df)
)


print(
    "\nClass distribution:"
)

print(
    final_test_df[
        "label"
    ].value_counts()
)


print(
    "\nFold × label distribution:"
)

display(
    pd.crosstab(
        final_test_df["fold"],
        final_test_df["label"]
    )
)


FINAL_TEST_RESULTS_PATH = (
    RESULTS_DIR /
    "qwen_counterbalanced_final_test_predictions.csv"
)


full_results = []


for index, row in final_test_df.iterrows():

    print(
        f"Processing review "
        f"{index + 1}/{len(final_test_df)}..."
    )


    result = score_review_counterbalanced(
        row["text"]
    )


    same_mapping_prediction = (
        result["prediction_mapping_1"]
        ==
        result["prediction_mapping_2"]
    )


    full_results.append({

        "review_id":
            row["review_id"],

        "gold_label":
            row["label"],

        "hotel":
            row["hotel"],

        "fold":
            row["fold"],

        "prediction":
            result["prediction"],

        "prediction_mapping_1":
            result["prediction_mapping_1"],

        "prediction_mapping_2":
            result["prediction_mapping_2"],

        "same_mapping_prediction":
            same_mapping_prediction,

        "deception_score":
            result["deception_score"],

        "deception_score_mapping_1":
            result["deception_score_mapping_1"],

        "deception_score_mapping_2":
            result["deception_score_mapping_2"]
    })


    # Save progress every 25 reviews.
    if (
        (index + 1) % 25 == 0
        or
        (index + 1) == len(final_test_df)
    ):

        pd.DataFrame(
            full_results
        ).to_csv(
            FINAL_TEST_RESULTS_PATH,
            index=False
        )

        print(
            f"Progress saved: "
            f"{index + 1} reviews"
        )


NameError: name 'pilot_df' is not defined

In [ ]:
FINAL_TEST_RESULTS_PATH = (
    RESULTS_DIR /
    "qwen_counterbalanced_final_test_predictions.csv"
)


full_results_df = pd.read_csv(
    FINAL_TEST_RESULTS_PATH
)


assert len(full_results_df) == 780
assert set(full_results_df["review_id"]).isdisjoint(
    pilot_ids
)


print(
    "Number of final-test predictions:",
    len(full_results_df)
)

print(
    "\nColumns:"
)

print(
    full_results_df.columns.tolist()
)


display(
    full_results_df.head()
)


In [ ]:
print(
    "Missing review IDs:",
    full_results_df[
        "review_id"
    ].isna().sum()
)


print(
    "Duplicate review IDs:",
    full_results_df[
        "review_id"
    ].duplicated().sum()
)


print(
    "Missing final predictions:",
    full_results_df[
        "prediction"
    ].isna().sum()
)


print(
    "\nGold-label distribution:"
)

print(
    full_results_df[
        "gold_label"
    ].value_counts()
)


print(
    "\nPrediction distribution:"
)

print(
    full_results_df[
        "prediction"
    ].value_counts()
)

In [ ]:
mapping_agreement_count = (
    full_results_df[
        "same_mapping_prediction"
    ].sum()
)


mapping_agreement_rate = (
    mapping_agreement_count
    / len(full_results_df)
)


print(
    "Same semantic prediction:",
    mapping_agreement_count,
    "/",
    len(full_results_df)
)


print(
    "Mapping agreement:",
    round(
        mapping_agreement_rate * 100,
        2
    ),
    "%"
)


print("\nMAPPING 1")

print(
    full_results_df[
        "prediction_mapping_1"
    ].value_counts()
)


print("\nMAPPING 2")

print(
    full_results_df[
        "prediction_mapping_2"
    ].value_counts()
)


print("\nFINAL COUNTERBALANCED PREDICTION")

print(
    full_results_df[
        "prediction"
    ].value_counts()
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


overall_accuracy = accuracy_score(
    full_results_df[
        "gold_label"
    ],
    full_results_df[
        "prediction"
    ]
)


# The untouched final test set is balanced
# (390 truthful, 390 deceptive), so random
# binary guessing has 50% expected accuracy.
chance_baseline = 0.50


# 95% Wilson confidence interval for accuracy.
n_test = len(
    full_results_df
)

n_correct = (
    full_results_df["gold_label"]
    ==
    full_results_df["prediction"]
).sum()

z = 1.959963984540054

p_hat = (
    n_correct
    / n_test
)

denominator = (
    1
    +
    z**2 / n_test
)

centre = (
    p_hat
    +
    z**2 / (2 * n_test)
) / denominator

half_width = (
    z
    *
    (
        (
            p_hat * (1 - p_hat)
            +
            z**2 / (4 * n_test)
        )
        / n_test
    ) ** 0.5
    / denominator
)

accuracy_ci_low = (
    centre
    - half_width
)

accuracy_ci_high = (
    centre
    + half_width
)


print(
    "Overall accuracy:",
    round(
        overall_accuracy,
        4
    )
)

print(
    "Chance baseline:",
    f"{chance_baseline:.0%}"
)

print(
    "95% Wilson CI:",
    f"[{accuracy_ci_low:.4f}, "
    f"{accuracy_ci_high:.4f}]"
)


print(
    "\nCLASSIFICATION REPORT\n"
)


print(
    classification_report(
        full_results_df[
            "gold_label"
        ],
        full_results_df[
            "prediction"
        ],
        labels=[
            "truthful",
            "deceptive"
        ],
        digits=4
    )
)


In [ ]:
full_confusion_table = pd.crosstab(
    full_results_df[
        "gold_label"
    ],
    full_results_df[
        "prediction"
    ],
    rownames=[
        "Gold label"
    ],
    colnames=[
        "Prediction"
    ]
)


display(
    full_confusion_table
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


# Overall accuracy.
overall_accuracy = accuracy_score(
    full_results_df["gold_label"],
    full_results_df["prediction"]
)


# Class-specific metrics.
precision, recall, f1, support = (
    precision_recall_fscore_support(
        full_results_df["gold_label"],
        full_results_df["prediction"],
        labels=[
            "truthful",
            "deceptive"
        ],
        zero_division=0
    )
)


performance_summary = pd.DataFrame({

    "label": [
        "truthful",
        "deceptive"
    ],

    "precision":
        precision,

    "recall":
        recall,

    "f1":
        f1,

    "support":
        support
})


print(
    "Overall accuracy:",
    round(overall_accuracy, 4)
)


print(
    "Uncertain predictions:",
    (
        full_results_df["prediction"]
        == "uncertain"
    ).sum()
)


display(
    performance_summary.round(4)
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay


fig, ax = plt.subplots(
    figsize=(6, 5)
)


ConfusionMatrixDisplay.from_predictions(
    full_results_df["gold_label"],
    full_results_df["prediction"],
    labels=[
        "truthful",
        "deceptive",
        "uncertain"
    ],
    ax=ax
)


ax.set_title(
    "Qwen3-4B-Instruct — Untouched Final Test"
)


fig.tight_layout()


fig.savefig(
    FIGURES_DIR /
    "qwen_final_test_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

In [ ]:
full_results_df["correct"] = (
    full_results_df["gold_label"]
    ==
    full_results_df["prediction"]
)


stability_summary = (
    full_results_df
    .groupby(
        "same_mapping_prediction"
    )
    .agg(

        n_reviews=(
            "review_id",
            "count"
        ),

        accuracy=(
            "correct",
            "mean"
        ),

        mean_abs_score=(
            "deception_score",
            lambda x: x.abs().mean()
        )
    )
    .reset_index()
)


stability_summary["percentage"] = (
    stability_summary["n_reviews"]
    / len(full_results_df)
    * 100
)


display(
    stability_summary.round(4)
)

In [ ]:
## 53. Evaluate Performance Across the Original Hotel Folds



In [ ]:
fold_summary = (
    full_results_df
    .groupby("fold")
    .agg(

        n_reviews=(
            "review_id",
            "count"
        ),

        accuracy=(
            "correct",
            "mean"
        ),

        deceptive_prediction_rate=(
            "prediction",
            lambda x: (
                x == "deceptive"
            ).mean()
        ),

        mapping_agreement=(
            "same_mapping_prediction",
            "mean"
        ),

        mean_abs_score=(
            "deception_score",
            lambda x: x.abs().mean()
        )
    )
    .reset_index()
)


display(
    fold_summary.round(4)
)

In [ ]:
fold_class_recall = (
    full_results_df
    .assign(
        correct_classification=(
            full_results_df["gold_label"]
            ==
            full_results_df["prediction"]
        )
    )
    .groupby(
        [
            "fold",
            "gold_label"
        ]
    )[
        "correct_classification"
    ]
    .mean()
    .unstack()
)


fold_class_recall.columns = [
    "deceptive_recall"
    if column == "deceptive"
    else "truthful_recall"
    for column in fold_class_recall.columns
]


display(
    fold_class_recall.round(4)
)

In [ ]:
import matplotlib.pyplot as plt


fig, ax = plt.subplots(
    figsize=(7, 5)
)


ax.bar(
    fold_summary["fold"],
    fold_summary["accuracy"]
)


ax.axhline(
    y=0.5,
    linestyle="--",
    linewidth=1
)


ax.set_ylim(
    0,
    1
)

ax.set_xlabel(
    "Original fold"
)

ax.set_ylabel(
    "Accuracy"
)

ax.set_title(
    "Qwen3-4B-Instruct Accuracy Across Hotel Folds"
)


fig.tight_layout()


fig.savefig(
    FIGURES_DIR /
    "qwen_accuracy_by_fold.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()

## Compare Decision Margin for Correct and Incorrect Predictions



In [ ]:
full_results_df["absolute_score"] = (
    full_results_df[
        "deception_score"
    ].abs()
)


margin_summary = (
    full_results_df
    .groupby("correct")[
        "absolute_score"
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std"
        ]
    )
)


display(
    margin_summary.round(4)
)

In [ ]:
full_results_df["score_quartile"] = pd.qcut(
    full_results_df["absolute_score"],
    q=4,
    labels=[
        "Q1 — weakest",
        "Q2",
        "Q3",
        "Q4 — strongest"
    ],
    duplicates="drop"
)


score_accuracy = (
    full_results_df
    .groupby(
        "score_quartile",
        observed=True
    )
    .agg(

        n_reviews=(
            "review_id",
            "count"
        ),

        mean_abs_score=(
            "absolute_score",
            "mean"
        ),

        accuracy=(
            "correct",
            "mean"
        )
    )
    .reset_index()
)


display(
    score_accuracy.round(4)
)

In [ ]:
# Dynamic final summary: values are computed from the
# untouched 780-review test set after rerunning inference.

prediction_counts = (
    full_results_df[
        "prediction"
    ]
    .value_counts()
)


mapping_agreement_rate = (
    full_results_df[
        "same_mapping_prediction"
    ]
    .mean()
)


print("FINAL EVALUATION SUMMARY")
print("------------------------")

print(
    "Final untouched test set:",
    len(full_results_df),
    "reviews"
)

print(
    "Chance baseline:",
    f"{chance_baseline:.0%}"
)

print(
    "Accuracy:",
    f"{overall_accuracy:.2%}"
)

print(
    "95% Wilson CI:",
    f"[{accuracy_ci_low:.2%}, "
    f"{accuracy_ci_high:.2%}]"
)

print(
    "Mapping agreement:",
    f"{mapping_agreement_rate:.2%}"
)

print(
    "\nPrediction distribution:"
)

print(
    prediction_counts
)

print(
    "\nInterpretation note:"
)

print(
    "The counterbalanced deception score is a "
    "decision margin, not a calibrated probability."
)

print(
    "Counterbalancing reduces sensitivity to label "
    "assignment/position but does not eliminate it."
)


In [ ]:
FINAL_LLM_RESULTS_PATH = (
    RESULTS_DIR /
    "qwen_counterbalanced_final_test_results.csv"
)


full_results_df.to_csv(
    FINAL_LLM_RESULTS_PATH,
    index=False
)


print("Final LLM results saved at:")
print(FINAL_LLM_RESULTS_PATH)

print(
    "\nFinal shape:",
    full_results_df.shape
)


----------

## Create a Fresh Sample for Cue-Guided Evaluation

An additional exploratory experiment is performed using a linguistically informed prompt containing explicit deception-related cues.

Because this prompt was developed after inspecting the original pilot, the 20 pilot reviews are excluded from this analysis.

A fresh stratified sample of 100 reviews is selected:
- 50 truthful;
- 50 deceptive;
- 20 reviews from each original fold;
- 10 reviews per class within each fold.

This experiment is treated as exploratory , is not a part of the project and is not used to replace the frozen zero-shot LLM baseline.

In [ ]:
pilot_ids = set(
    pilot_df["review_id"]
)


exploratory_pool = df_clean[
    ~df_clean["review_id"].isin(
        pilot_ids
    )
]


cue_sample_df = (
    exploratory_pool
    .groupby(
        ["fold", "label"],
        group_keys=False
    )
    .sample(
        n=10,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)


print(
    "Sample size:",
    len(cue_sample_df)
)


print("\nClass distribution:")

print(
    cue_sample_df[
        "label"
    ].value_counts()
)


print("\nFold × label distribution:")

display(
    pd.crosstab(
        cue_sample_df["fold"],
        cue_sample_df["label"]
    )
)

In [ ]:
def build_cue_guided_prompt(review_text):

    return f"""
You are an expert evaluator of online reviews.

Your task is to classify the following hotel review into exactly one of two categories:

- TRUTHFUL: The review appears to describe a genuine hotel experience and does not show substantial evidence of intentional deception.
- DECEPTIVE: The review appears intentionally fabricated, misleading, or designed to create a false impression of a hotel experience.

Your judgment should be based on the linguistic characteristics of the review.

IMPORTANT:
The cues listed below are indicators to consider, not deterministic rules.
No single cue is sufficient to classify a review as truthful or deceptive.
Evaluate how multiple cues interact and whether they form a coherent overall pattern.

When evaluating the review, pay particular attention to the following characteristics.

### 1. Review characteristics

**Argument quality**
Consider the quality, coherence, and persuasiveness of the review's arguments.
Well-developed and coherent reasoning may support perceived genuineness, but persuasive language alone is not sufficient evidence of truthfulness.

**Text length**
Notice whether the review is unusually short or unusually long.
Extreme length may raise suspicion, but length alone must not determine the classification.

**Two-sidedness**
Consider whether the reviewer naturally discusses both positive and negative aspects of the experience.
A balanced account containing advantages and disadvantages may increase perceived credibility.

**Affective intensity**
Consider the emotional intensity of the language.
Very strong emotional language may reduce credibility, especially when it appears disproportionate or strategically persuasive.
However, emotion alone is not evidence of deception.

### 2. Textual characteristics

**Specificity and vagueness**
Look for concrete descriptions, specific information, names, places, actions, or events.
Vagueness, omitted information, and generic descriptions may raise suspicion.

However, do not assume that specific details automatically make a review truthful.
Specific details can also be fabricated.

**Grammatical and spelling errors**
Notice grammatical, syntactic, or spelling errors.
Such errors may influence perceived credibility, but they must not be treated as direct evidence of deception.

**Personal references and subjectivity**
Consider the use of first-person pronouns and other personal references.
Frequent first-person referencing may sometimes reduce perceived genuineness, although greater subjectivity can also make an account appear more personally grounded.
Interpret this cue only in combination with the rest of the review.

**Contextual embedding**
Look for whether the claimed experience is naturally embedded in a specific temporal, spatial, or situational context.
References to when, where, why, or under what circumstances something happened may support perceived genuineness.
Fabricated reviews may sometimes omit these connections.

**Level of detail**
Evaluate not only whether details are present, but whether the amount and type of detail appear natural for the experience being described.
A complete absence of detail may be suspicious, but an excessive or unnaturally elaborate level of detail may also indicate fabrication.

**Flattering and stylistic cues**
Notice excessive praise, promotional language, exclamation marks, emoticons, or other stylistic devices that appear designed to persuade the reader.
Strongly flattering or promotional language may indicate deceptive intent, particularly when it is not grounded in concrete experience.

### 3. Overall evaluation

Do not simply count the number of truthful or deceptive cues.

Instead, consider:
- whether the cues are mutually consistent;
- whether details are naturally connected to a plausible personal experience;
- whether the review is contextually grounded;
- whether praise or criticism is supported by experience-based information;
- whether the language appears primarily experience-driven or strategically persuasive;
- whether apparently truthful cues could themselves have been deliberately fabricated.

Distinguish between unusual writing and intentional deception.

If the evidence is mixed or ambiguous, still select the best-supported label, but assign lower confidence.

Review:
\"\"\"{review_text}\"\"\"

Return the result in exactly this format:

LABEL: <TRUTHFUL or DECEPTIVE>
CONFIDENCE: <number from 0.00 to 1.00>
EVIDENCE:
- <brief reason referring to a relevant linguistic cue>
- <brief reason referring to a relevant linguistic cue>
- <brief reason referring to a relevant linguistic cue>
"""

In [ ]:
def classify_review_cue_guided(review_text):

    prompt = build_cue_guided_prompt(
        review_text
    )


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)


    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False
        )


    generated_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]


    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


    return response.strip()

In [ ]:
import re


def parse_expert_output(model_output):

    # Extract predicted label.
    label_match = re.search(
        r"LABEL:\s*(TRUTHFUL|DECEPTIVE)",
        model_output,
        re.IGNORECASE
    )


    # Extract confidence.
    confidence_match = re.search(
        r"CONFIDENCE:\s*([01](?:\.\d+)?)",
        model_output,
        re.IGNORECASE
    )


    # Extract everything after EVIDENCE.
    evidence_match = re.search(
        r"EVIDENCE:\s*(.*)",
        model_output,
        re.IGNORECASE | re.DOTALL
    )


    label = None
    confidence = None
    evidence = None


    if label_match:

        label = (
            label_match
            .group(1)
            .lower()
        )


    if confidence_match:

        confidence = float(
            confidence_match.group(1)
        )


    if evidence_match:

        evidence = (
            evidence_match
            .group(1)
            .strip()
        )


    return {
        "prediction": label,
        "confidence": confidence,
        "evidence": evidence
    }

In [ ]:
CUE_RESULTS_PATH = (
    RESULTS_DIR /
    "qwen_cue_guided_100_reviews.csv"
)


cue_results = []


for index, row in cue_sample_df.iterrows():

    print(
        f"Processing review "
        f"{index + 1}/{len(cue_sample_df)}..."
    )


    raw_output = classify_review_cue_guided(
        row["text"]
    )


    parsed = parse_expert_output(
        raw_output
    )


    cue_results.append({

        "review_id":
            row["review_id"],

        "gold_label":
            row["label"],

        "hotel":
            row["hotel"],

        "fold":
            row["fold"],

        "prediction":
            parsed["prediction"],

        "confidence":
            parsed["confidence"],

        "evidence":
            parsed["evidence"],

        "raw_output":
            raw_output
    })


    if (
        (index + 1) % 10 == 0
        or
        (index + 1) == len(cue_sample_df)
    ):

        pd.DataFrame(
            cue_results
        ).to_csv(
            CUE_RESULTS_PATH,
            index=False
        )

        print(
            f"Progress saved: "
            f"{index + 1} reviews"
        )

In [ ]:
cue_results_df = pd.DataFrame(
    cue_results
)


print(
    "Missing predictions:",
    cue_results_df[
        "prediction"
    ].isna().sum()
)


print(
    "Missing confidence:",
    cue_results_df[
        "confidence"
    ].isna().sum()
)


print("\nPREDICTION DISTRIBUTION")

print(
    cue_results_df[
        "prediction"
    ].value_counts()
)


cue_results_df["correct"] = (
    cue_results_df["gold_label"]
    ==
    cue_results_df["prediction"]
)


cue_accuracy = (
    cue_results_df["correct"]
    .mean()
)


print(
    "\nAccuracy:",
    round(
        cue_accuracy,
        4
    )
)


print("\nCONFUSION TABLE")

display(
    pd.crosstab(
        cue_results_df[
            "gold_label"
        ],
        cue_results_df[
            "prediction"
        ],
        rownames=[
            "Gold label"
        ],
        colnames=[
            "Prediction"
        ]
    )
)